# Phase-1: Baseline CNN-LSTM Model Training

**Goal:** Train and evaluate the numeric-only baseline model.

**Steps:**
1. Load extracted features
2. Prepare train/test split (70/30 time-based)
3. Build CNN-LSTM model
4. Train with early stopping
5. Evaluate performance (MAE, RMSE, MAPE)
6. Visualize predictions
7. Analyze residuals

In [ ]:
# --- Portable setup (Windows / macOS / Linux) — do not hardcode paths ---
import importlib.util
from pathlib import Path

def _load_notebook_setup():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "notebooks" / "notebook_setup.py",
        cwd / "notebook_setup.py",
    ]
    if cwd.name == "notebooks":
        candidates.insert(0, cwd / "notebook_setup.py")
    for fp in candidates:
        if fp.is_file():
            spec = importlib.util.spec_from_file_location("_notebook_setup", fp)
            mod = importlib.util.module_from_spec(spec)
            assert spec.loader is not None
            spec.loader.exec_module(mod)
            return mod
    raise FileNotFoundError(
        "Missing notebooks/notebook_setup.py. Unzip the full project, open a terminal, "
        "cd into the repo root, then run: jupyter notebook"
    )

_ns = _load_notebook_setup()
PROJECT_ROOT = _ns.setup()
print(f"PROJECT_ROOT = {PROJECT_ROOT}")


import sys
import os



import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from utils import load_config, plot_style_setup, calculate_metrics, print_metrics
from baseline_model import BaselineTrainer

plot_style_setup()
config = load_config('config.yaml')

print("✓ Imports complete")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Load Features

In [ ]:
import os

# Load features CSV
features_path = os.path.join('../', config['paths']['features'], 'features_B0005.csv')
features_df = pd.read_csv(features_path)

print(f"Loaded features: {features_df.shape}")
print(f"\nFeature columns:")
for col in features_df.columns:
    print(f"  - {col}")

# Display first few rows
features_df.head()

## 2. Feature Statistics

In [ ]:
# Summary statistics
print("\nFeature Statistics:")
features_df.describe()

## 3. Visualize SOH Degradation

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(features_df['cycle_number'], features_df['soh'], 'o-', linewidth=2, markersize=4)
ax.axhline(y=100, color='g', linestyle='--', alpha=0.5, label='New Battery (100%)')
ax.axhline(y=80, color='r', linestyle='--', alpha=0.5, label='EOL Threshold (80%)')
ax.set_xlabel('Cycle Number', fontsize=12)
ax.set_ylabel('SOH (%)', fontsize=12)
ax.set_title('State of Health (SOH) Degradation', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nSOH Range: [{features_df['soh'].min():.2f}, {features_df['soh'].max():.2f}]%")

## 4. Train/Test Split Visualization

In [ ]:
# Split point
train_ratio = config['split']['train_ratio']
n_train = int(len(features_df) * train_ratio)

fig, ax = plt.subplots(figsize=(12, 6))

# Plot train/test regions
ax.plot(features_df['cycle_number'][:n_train], features_df['soh'][:n_train], 
        'o-', linewidth=2, markersize=4, label='Train', color='blue')
ax.plot(features_df['cycle_number'][n_train:], features_df['soh'][n_train:], 
        'o-', linewidth=2, markersize=4, label='Test', color='orange')

ax.axvline(x=n_train, color='r', linestyle='--', linewidth=2, label=f'Split (70/30)')
ax.set_xlabel('Cycle Number', fontsize=12)
ax.set_ylabel('SOH (%)', fontsize=12)
ax.set_title('Time-Based Train/Test Split', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTrain cycles: 0-{n_train-1} ({n_train} cycles)")
print(f"Test cycles: {n_train}-{len(features_df)-1} ({len(features_df)-n_train} cycles)")

## 5. Build and Train Model

In [ ]:
# Initialize trainer
trainer = BaselineTrainer(config)

# Prepare data
train_loader, test_loader = trainer.prepare_data(features_df)

# Build model
input_size = len(trainer.feature_cols)
trainer.build_model(input_size)

In [ ]:
# Train model
trainer.train(train_loader, test_loader)

## 6. Training History

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

epochs = range(1, len(trainer.train_losses) + 1)
ax.plot(epochs, trainer.train_losses, 'o-', label='Train Loss', linewidth=2, markersize=4)
ax.plot(epochs, trainer.val_losses, 'o-', label='Validation Loss', linewidth=2, markersize=4)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss (MSE)', fontsize=12)
ax.set_title('Training History', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal Training Loss: {trainer.train_losses[-1]:.4f}")
print(f"Final Validation Loss: {trainer.val_losses[-1]:.4f}")

## 7. Model Evaluation

In [ ]:
# Evaluate on test set
metrics, predictions, actuals = trainer.evaluate(test_loader)

print_metrics(metrics, title="Test Set Performance")

## 8. Predictions vs Actuals

In [ ]:
# Get test cycle numbers (accounting for lookback)
lookback = config['sequence']['lookback_cycles']
test_cycle_start = n_train + lookback
test_cycle_numbers = features_df['cycle_number'][test_cycle_start:].values

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Predictions vs Actuals over time
axes[0].plot(test_cycle_numbers, actuals, 'o-', label='Actual SOH', linewidth=2, markersize=5)
axes[0].plot(test_cycle_numbers, predictions, 's-', label='Predicted SOH', linewidth=2, markersize=5, alpha=0.7)
axes[0].set_xlabel('Cycle Number', fontsize=12)
axes[0].set_ylabel('SOH (%)', fontsize=12)
axes[0].set_title('Predictions vs Actuals', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Scatter plot (perfect prediction line)
axes[1].scatter(actuals, predictions, alpha=0.6, s=50)
# Perfect prediction line
min_val = min(actuals.min(), predictions.min())
max_val = max(actuals.max(), predictions.max())
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual SOH (%)', fontsize=12)
axes[1].set_ylabel('Predicted SOH (%)', fontsize=12)
axes[1].set_title('Prediction Scatter Plot', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_aspect('equal', adjustable='box')

plt.tight_layout()
plt.show()

## 9. Residual Analysis

In [ ]:
# Compute residuals
residuals = actuals - predictions

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Residuals over cycles
axes[0].plot(test_cycle_numbers, residuals, 'o-', linewidth=2, markersize=5)
axes[0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0].set_xlabel('Cycle Number', fontsize=12)
axes[0].set_ylabel('Residual (Actual - Predicted) %', fontsize=12)
axes[0].set_title('Prediction Residuals', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Plot 2: Residual distribution
axes[1].hist(residuals, bins=20, edgecolor='black', alpha=0.7)
axes[1].axvline(x=0, color='r', linestyle='--', linewidth=2)
axes[1].axvline(x=np.mean(residuals), color='g', linestyle='--', linewidth=2, label=f'Mean: {np.mean(residuals):.2f}%')
axes[1].set_xlabel('Residual (%)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Residual Distribution', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nResidual Statistics:")
print(f"  Mean: {np.mean(residuals):.4f}%")
print(f"  Std: {np.std(residuals):.4f}%")
print(f"  Range: [{np.min(residuals):.4f}, {np.max(residuals):.4f}]%")

## 10. Save Model

In [ ]:
# Save trained model
model_dir = os.path.join('../', config['paths']['models'])
trainer.save_model(model_dir, model_name='baseline_cnn_lstm_B0005')

## Summary

**Phase-1 Baseline Results:**

**Model:** CNN-LSTM (Numeric features only)
**Features:** SOH, Capacity, ICA peaks, DCIR, Temperature stats, Voltage/Current stats

**Performance Metrics:**
- Check the printed metrics above

**Key Observations:**
1. Baseline model captures overall degradation trend
2. Physics-guided features (ICA, DCIR) contribute to predictions
3. Residuals show systematic patterns that could be improved

**Next Steps (Phase-2):**
1. Add semantic features (text descriptions)
2. Implement LLM embeddings
3. Build multimodal fusion architecture
4. Compare: Numeric-only vs Numeric+Semantic
5. Add explainability (SHAP, attention visualization)

**This baseline is the reference for Phase-2 improvements!**